In [1]:
import sys
print(sys.executable)

c:\Apps\MiniConda\envs\ml_edu\python.exe


In [2]:
import pandas as pd

from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones



# UCI 데이터 수집

In [3]:
import requests
import zipfile
from pathlib import Path

ZIP_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip" 
ZIP_DIR = "./data"

In [4]:
def unzip_from_response(zip_url=ZIP_URL, zip_dir=ZIP_DIR) -> str :
    # 디렉토리 생성, 있으면 무시
    # 데이터 디렉토리 생성
    Path(zip_dir).mkdir(parents=True, exist_ok=True)

    r = requests.get(zip_url, timeout=30)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall(zip_dir)

    print("압축 해제 완료:", zip_dir)


# urllib.request.urlretrieve(url, zip_path)

In [5]:
from pathlib import Path

for file in Path(ZIP_DIR).rglob("*"):
    print(file, '(디렉토리)' if file.is_dir() else "")

data\UCI HAR Dataset (디렉토리)
data\__MACOSX (디렉토리)
data\UCI HAR Dataset\.DS_Store 
data\UCI HAR Dataset\activity_labels.txt 
data\UCI HAR Dataset\features.txt 
data\UCI HAR Dataset\features_info.txt 
data\UCI HAR Dataset\README.txt 
data\UCI HAR Dataset\test (디렉토리)
data\UCI HAR Dataset\train (디렉토리)
data\UCI HAR Dataset\test\Inertial Signals (디렉토리)
data\UCI HAR Dataset\test\subject_test.txt 
data\UCI HAR Dataset\test\X_test.txt 
data\UCI HAR Dataset\test\y_test.txt 
data\UCI HAR Dataset\test\Inertial Signals\body_acc_x_test.txt 
data\UCI HAR Dataset\test\Inertial Signals\body_acc_y_test.txt 
data\UCI HAR Dataset\test\Inertial Signals\body_acc_z_test.txt 
data\UCI HAR Dataset\test\Inertial Signals\body_gyro_x_test.txt 
data\UCI HAR Dataset\test\Inertial Signals\body_gyro_y_test.txt 
data\UCI HAR Dataset\test\Inertial Signals\body_gyro_z_test.txt 
data\UCI HAR Dataset\test\Inertial Signals\total_acc_x_test.txt 
data\UCI HAR Dataset\test\Inertial Signals\total_acc_y_test.txt 
data\UCI HAR Da

# get_human_dataset()

In [ ]:
def get_human_dataset(base_path='./data/UCI HAR Dataset/'):

    # 피처 이름 로딩

    feature_path = base_path + 'features.txt'

    feature_name_df = pd.read_csv(feature_path, sep='\s+', header=None, names=['column_index', 'column_name'])



    # 중복 피처명 처리

    new_feature_name_df = get_new_feature_name_df(feature_name_df)

    feature_names = new_feature_name_df['column_name'].tolist()



    # 학습/테스트 데이터 로딩

    X_train = pd.read_csv(base_path + 'train/X_train.txt', sep='\s+', names=feature_names)

    X_test = pd.read_csv(base_path + 'test/X_test.txt', sep='\s+', names=feature_names)



    y_train = pd.read_csv(base_path + 'train/y_train.txt', sep='\s+', header=None, names=['action'])

    y_test = pd.read_csv(base_path + 'test/y_test.txt', sep='\s+', header=None, names=['action'])

    return X_train, X_test, y_train, y_test

# plot_feature_importances()

In [ ]:
# 피처 중요도 출력
import matplotlib.pyplot as plt
import numpy as np

def plot_feature_importances(model, feature_names, top_n=20):
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:top_n]
    plt.figure(figsize=(10, top_n * 0.4))  # 피처 수에 따라 자동 높이 조절

    plt.barh(range(top_n), importances[indices][::-1], align='center')
    plt.yticks(range(top_n), np.array(feature_names)[indices][::-1])
    plt.xlabel('Feature Importance')
    plt.ylabel('Feature')
    plt.title(f'Top {top_n} Feature Importances')
    plt.tight_layout()
    plt.show()



# clean_feature_name()

In [ ]:
import re
def clean_feature_name(name):
    # 특수문자 제거: 괄호, 콤마, 하이픈 등
    return re.sub('[^A-Za-z0-9_]+', '_', name)

# get_new_feature_name_df()

In [ ]:
def get_new_feature_name_df(old_feature_name_df):
    feature_dup_df = old_feature_name_df.groupby('column_name').cumcount()
    new_feature_name_df = old_feature_name_df.copy()
    new_feature_name_df = new_feature_name_df.reset_index(drop=True)  # 인덱스 리셋
    feature_dup_df = feature_dup_df.reset_index(drop=True)            # 인덱스 리셋
    new_feature_name_df['column_name'] = new_feature_name_df['column_name'] + "_" + feature_dup_df.astype(str)
    return new_feature_name_df

# 데이터 분할

In [ ]:
X_train, X_test, y_train, y_test = get_human_dataset(base_path='./data/UCI HAR Dataset/')

print(X_train.shape, X_test.shape)
print(y_train['action'].value_counts())

# 랜덤포레스트

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_clf = RandomForestClassifier(random_state=0, max_depth=8)
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)
accuracy_score(y_test, rf_pred)


# 랜덤포레스트 GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'max_depth': [8,16,24, 32],
    'min_samples_split' : [2,8,16],
    'min_samples_leaf': [1,6,12]
}

rf_clf = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
grid_cv = GridSearchCV(rf_clf, param_grid=params, cv=2, n_jobs=-1)
grid_cv.fit(X_train, y_train)
grid_cv.best_params_, grid_cv.best_score_

In [ ]:
# 모델 성능 평가
best_model = grid_cv.best_estimator_
best_pred = best_model.predict(X_test)
accuracy_score(y_test, best_pred)

In [ ]:
plot_feature_importances(best_model, feature_names=X_train.columns)

# GBM Gradient Boosting Macine

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb_clf = GradientBoostingClassifier(random_state=0)
gb_clf.fit(X_train, y_train)
gb_pred = gb_clf.predict(X_test)
accuracy_score(y_test, gb_pred)

# XGBOOST

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)  # Series → 1D array
y_test_encoded = le.transform(y_test)

In [ ]:
xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.1,
    max_depth=3,
    early_stopping_rounds=40,   # 여기로 이동
    eval_metric='mlogloss',      # 여기로 이동
    use_label_encoder=False
)

evals = [(X_test, y_test_encoded)]
xgb.fit(X_train, y_train_encoded, eval_set=evals, verbose=True)

xgb_pred = xgb.predict(X_test)

In [ ]:
print(classification_report(y_test_encoded, xgb_pred))

# LGBM

In [ ]:
from lightgbm import early_stopping, log_evaluation
from lightgbm import LGBMClassifier

evals = [(X_test, y_test_encoded)]
lgb = LGBMClassifier(n_estimators=400, objective='multiclass',
                     num_class=len(np.unique(y_train_encoded)))

lgb.fit(X_train, y_train_encoded,
        eval_set=evals,
        callbacks=[early_stopping(40), log_evaluation(period=10)])  # 10 iter마다 출력

lgb_pred = lgb.predict(X_test)

In [ ]:
print(classification_report(y_test_encoded, lgb_pred))